# Predictive Maintenance - NASA C-MAPSS (XGBoost)
This project is inspired by Honeywell's predictive maintenance problem.
The goal is to predict engine failures in advance using multiple sensor readings.
We approach this by predicting the Remaining Useful Life (RUL).

### 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import os

# import our helper functions
from utils import calculate_rmse, calculate_mae, drop_constant_columns, remove_highly_correlated_features

# suppress warnings to keep the notebook clean
import warnings
warnings.filterwarnings('ignore')

### 2. Load Data

In [ ]:
# using multiple sensor readings for prediction
# Set column names
columns = ['unit', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + [f'sensor_{i}' for i in range(1, 22)]

# Read the files from the dataset folder
train_data = pd.read_csv('dataset/train_FD001.txt', sep='\s+', header=None, names=columns)
test_data = pd.read_csv('dataset/test_FD001.txt', sep='\s+', header=None, names=columns)
rul_data = pd.read_csv('dataset/RUL_FD001.txt', sep='\s+', header=None, names=['RUL_actual'])

print("Training data shape:", train_data.shape)
print("Testing data shape:", test_data.shape)

### 3. RUL Calculation
For the train data, the Remaining Useful Life (RUL) is the max cycle minus the current cycle.
For the test data, the true RUL is provided only for the last cycle of each engine in `RUL_FD001.txt`.

In [ ]:
# Calculate RUL for training data
train_max = train_data.groupby('unit')['cycle'].max().reset_index()
train_max.columns = ['unit', 'max_cycle']
train_data = train_data.merge(train_max, on='unit', how='left')
train_data['RUL'] = train_data['max_cycle'] - train_data['cycle']

# Calculate RUL for testing data
rul_data['unit'] = rul_data.index + 1
test_max = test_data.groupby('unit')['cycle'].max().reset_index()
test_max.columns = ['unit', 'max_cycle']
test_data = test_data.merge(test_max, on='unit', how='left')
test_data = test_data.merge(rul_data, on='unit', how='left')

# The actual RUL at any given cycle in the test set
test_data['RUL'] = test_data['RUL_actual'] + (test_data['max_cycle'] - test_data['cycle'])

# drop the extra target column
test_data.drop(columns=['RUL_actual'], inplace=True)

### 4. Feature Engineering
We will add a normalized cycle feature and a rolling mean for the sensors with a window of 5.

In [ ]:
# normalized cycle feature
train_data['norm_cycle'] = train_data['cycle'] / train_data['max_cycle']
test_data['norm_cycle'] = test_data['cycle'] / test_data['max_cycle']

# get all the sensor column names
sensors = [f'sensor_{i}' for i in range(1, 22)]

# calculate rolling means
train_roll = train_data.groupby('unit')[sensors].rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)
test_roll = test_data.groupby('unit')[sensors].rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)

# rename the rolling columns
train_roll.columns = [f'{s}_roll' for s in sensors]
test_roll.columns = [f'{s}_roll' for s in sensors]

# add them back to the original dataframes
train_data = pd.concat([train_data, train_roll], axis=1)
test_data = pd.concat([test_data, test_roll], axis=1)
# combining features from different sensors (feature fusion)

### 5. Data Cleaning
We use our helper functions from `utils.py` to drop constant columns and highly correlated features so the model doesn't overfit to redundant data.

In [ ]:
# Drop constant columns
train_data, test_data = drop_constant_columns(train_data, test_data)

# Drop highly correlated columns
exclude_from_corr = ['unit', 'cycle', 'max_cycle', 'RUL']
train_data, test_data = remove_highly_correlated_features(train_data, test_data, exclude_from_corr)

### 6. Feature Scaling
Scaling is important. We fit the scaler on the training data only.

In [ ]:
# Get the final list of features we will use for training
final_features = [c for c in train_data.columns if c not in exclude_from_corr]

# scale the features
scaler = StandardScaler()
train_data[final_features] = scaler.fit_transform(train_data[final_features])
test_data[final_features] = scaler.transform(test_data[final_features])

# prepare X and y
X_train = train_data[final_features]
y_train = train_data['RUL']
X_test = test_data[final_features]
y_test = test_data['RUL']

### 7. Baseline
Let's make a simple baseline by just predicting the mean RUL of the training set.

In [ ]:
mean_rul = y_train.mean()
baseline_preds = np.full_like(y_test, fill_value=mean_rul)

baseline_rmse = calculate_rmse(y_test, baseline_preds)
baseline_mae = calculate_mae(y_test, baseline_preds)

print("Baseline RMSE:", round(baseline_rmse, 2))
print("Baseline MAE:", round(baseline_mae, 2))

### 8. Model Training
Now we train the XGBoost Regressor model.
We use XGBoost here to see if we can get better performance.

In [ ]:
xgb_model = XGBRegressor(
    random_state=42, 
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

### 9. Evaluation
Let's see how our XGBoost model performed compared to the baseline.

In [ ]:
# predicting RUL to estimate failure in advance
y_pred = xgb_model.predict(X_test)

xgb_rmse = calculate_rmse(y_test, y_pred)
xgb_mae = calculate_mae(y_test, y_pred)

print("XGBoost RMSE:", round(xgb_rmse, 2))
print("XGBoost MAE:", round(xgb_mae, 2))

### 10. Visualization
Plotting the actual vs predicted RUL for the last cycle of each engine.

In [ ]:
# add predictions back to group by unit
test_data['predicted_RUL'] = y_pred

# get the last cycle for each engine
last_cycles = test_data.groupby('unit').last()

# plot the results
plt.figure(figsize=(10, 5))
plt.plot(last_cycles.index, last_cycles['RUL'], label='Actual RUL', marker='o')
plt.plot(last_cycles.index, last_cycles['predicted_RUL'], label='Predicted RUL', marker='x')

plt.title('Actual vs Predicted RUL (End of Life)')
plt.xlabel('Engine Unit')
plt.ylabel('RUL')
plt.legend()
plt.grid(True)

# make sure results folder exists and save plot
os.makedirs('results', exist_ok=True)
plt.savefig('results/xgboost_actual_vs_predicted.png')
plt.show()